In [9]:
import os
import re
import pickle as pkl
import ast
from collections import Counter
from typing import List, Tuple, Any
from tqdm import tqdm
import time

import pandas as pd
import numpy as np
import calibration as cal

import torch

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')
pal = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [10]:
from llm_unsupervised_conf.metrics import *
from llm_unsupervised_conf.math_utils import stack_embeddings
from llm_unsupervised_conf.calibration import fit_predict_prob_models
from llm_unsupervised_conf.plots import plot_reliability, plot_avg_and_worstcase_by_method, plot_method_comparisons, METHODS_MAP_SHORT

In [11]:
from llm_unsupervised_conf.utils import maybe_add_verbal_conf_row, load_out_df, set_seed

In [12]:

def run_exp(
    dataset,
    model,
    n=1000,
    k_train=100,
    temp_train=0.7,
    temp_test=0.6,
    prob_models=("ridge_clip",),   # <-- pass a list/tuple of the methods above
    include_verbal_conf=True,
    random_state=42,
    n_bins=12,
    embedding_text="question_response",
    drop_bad_rows=True,
    test_prop=0.6,
    verbose=False,
    include_alt_ablation=True,
    include_question_ablation=True
):
    """
    prob_models controls which embedding->prob models to include.
    Example:
      prob_models=["ridge_clip","ridge_logit","hgb_logit","mlp_logit","isotonic_on_ridge"]
    """

    set_seed(random_state)

    model_name = model.split("/")[-1]

    try:
        out_df, df_save_path = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text, drop_bad_rows)
    except:
        return None

    # print(df_save_path)
    if include_alt_ablation:
        ablation_df, _ = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text="question_response_gemma", drop_bad_rows=True)
        out_df["alt_embeddings"] = ablation_df["embeddings"]

    if include_question_ablation:
        ablation_df, _ = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text="question", drop_bad_rows=True)
        out_df["question_embeddings"] = ablation_df["embeddings"]

    if verbose or random_state == 0:
        print(dataset, model, "no rows", len(out_df))

    # --------------------
    # Split
    # --------------------
    train_df, test_df = train_test_split(
        out_df,
        test_size=test_prop,
        random_state=random_state,
        shuffle=True,
    )

    X_train = stack_embeddings(train_df, "embeddings")
    y_train = train_df["consistency"].astype(np.float32).to_numpy()  # target in [0,1]

    X_test = stack_embeddings(test_df, "embeddings")

    # These are for evaluation
    con_scores = test_df["consistency"].to_numpy(dtype=float)
    correct = test_df["correct"].to_numpy(dtype=int)

    logprobs = test_df["avg_logprobs"].to_numpy(dtype=float)
    ans_logprobs = test_df["ans_logprobs"].to_numpy(dtype=float)
    lp = np.exp(logprobs)
    alp = np.exp(ans_logprobs)

    test_df["token_probs"] = lp
    test_df["ans_token_probs"] = alp

    ########################
    # Ours
    ########################
    prob_models = list(prob_models) if prob_models is not None else []
    preds = fit_predict_prob_models(
        X_train=X_train,
        y_train_prob=y_train,
        X_test=X_test,
        methods=prob_models,
        random_state=random_state,
    )

    for method_name, pred_probs in preds.items():
        test_df[method_name] = pred_probs

    ########################
    # Verbal Confidence
    ########################
    vc_scores = None
    if include_verbal_conf:
        _, vc_scores = maybe_add_verbal_conf_row(
            [], 
            df_save_path, 
            test_df, 
            correct, 
            stem_idx=6
        )
        test_df["vc"] = vc_scores

    ########################
    # Alt. Model Embeddings
    ########################

    if include_alt_ablation:
        X_train = stack_embeddings(train_df, "alt_embeddings")
        X_test = stack_embeddings(test_df, "alt_embeddings")
        abl_preds = fit_predict_prob_models(
            X_train=X_train,
            y_train_prob=y_train,
            X_test=X_test,
            methods=prob_models,
            random_state=random_state,
        )
        for method_name, abl_probs in abl_preds.items():
            test_df[f"blackbox_{method_name}"] = abl_probs

    ########################
    # Question Only Embeddings
    ########################

    if include_alt_ablation:
        X_train = stack_embeddings(train_df, "question_embeddings")
        X_test = stack_embeddings(test_df, "question_embeddings")
        abl_preds = fit_predict_prob_models(
            X_train=X_train,
            y_train_prob=y_train,
            X_test=X_test,
            methods=prob_models,
            random_state=random_state,
        )
        for method_name, abl_probs in abl_preds.items():
            test_df[f"question_only_{method_name}"] = abl_probs

    test_df["model"] = model
    test_df["dataset"] = dataset

    # print("-*-*"*16)
    # print("Exp DF")
    # print("No. rows", len(test_df))
    # print("Columns:", list(test_df.columns))
    # display(test_df.head())

    return test_df


In [13]:
def run_all_exps(
    models, 
    datasets, 
    exp_key=None, 
    n_trials=10, 
    n_bins=12, 
    test_prop=0.6,
    embedding_text="question_response",
    prob_models = ["ridge_clip","split_isotonic_on_ridge","split_isotonic_on_ridge_nrt"],
):

    full_df = []
    for dataset in datasets:
        print(f"[status] running {dataset}")
        start_time = time.time()
        for model in models:
            print(f"[status] running {model}")
            for seed in range(n_trials):
                seed = seed+35
                df = run_exp(
                    dataset=dataset,
                    model=model,
                    random_state=seed,
                    n_bins=n_bins,
                    test_prop=test_prop,
                    embedding_text=embedding_text,
                    prob_models=prob_models
                )
                if df is None:
                    continue
                df["seed"] = seed
                full_df.append(df)
                
        print("--- %s seconds ---" % (time.time() - start_time))

    full_df = pd.concat(full_df)
    return full_df
    


In [14]:
models = [
    "Qwen/Qwen3-0.6B", 
    "Qwen/Qwen3-1.7B", 
    "Qwen/Qwen3-4B-Thinking-2507", 
    "Qwen/Qwen3-8B",
    "Qwen/Qwen3-14B",
    "nvidia/OpenReasoning-Nemotron-7B",
    "nvidia/Nemotron-Cascade-8B-Thinking",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
]
datasets = [
    "gsm8k",
    "polymath",
    "trivia_qa",
    "sciq",
    "webq"
]
exp_key = "failures"
n_trials = 4

full_df = run_all_exps(models, datasets, exp_key, n_trials)

[status] running gsm8k
[status] running Qwen/Qwen3-0.6B
[status] running Qwen/Qwen3-1.7B
[status] running Qwen/Qwen3-4B-Thinking-2507
[status] running Qwen/Qwen3-8B
[status] running Qwen/Qwen3-14B
[status] running nvidia/OpenReasoning-Nemotron-7B
[status] running nvidia/Nemotron-Cascade-8B-Thinking
[status] running deepseek-ai/DeepSeek-R1-Distill-Llama-8B
[status] running deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
--- 26.59351634979248 seconds ---
[status] running polymath
[status] running Qwen/Qwen3-0.6B
[status] running Qwen/Qwen3-1.7B
[status] running Qwen/Qwen3-4B-Thinking-2507
[status] running Qwen/Qwen3-8B
[status] running Qwen/Qwen3-14B
[status] running nvidia/OpenReasoning-Nemotron-7B
[status] running nvidia/Nemotron-Cascade-8B-Thinking
[status] running deepseek-ai/DeepSeek-R1-Distill-Llama-8B
[status] running deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
--- 27.877212285995483 seconds ---
[status] running trivia_qa
[status] running Qwen/Qwen3-0.6B
[status] running Qwen/Qwen3-1.7B
[stat

In [15]:
full_df

,id,question,ground_truth,answer,response,avg_logprobs,ans_logprobs,num_output_tokens,correct,consistency,...,vc,blackbox_ridge_clip,blackbox_split_isotonic_on_ridge,blackbox_split_isotonic_on_ridge_nrt,question_only_ridge_clip,question_only_split_isotonic_on_ridge,question_only_split_isotonic_on_ridge_nrt,model,dataset,seed
567,568,John pays for half the cost of raising a child...,265000,530000,"<think>\nOkay, let's try to figure out how muc...",-0.254461,-0.000925,3130,0,0.45,...,1.00,0.402894,0.517143,0.726000,0.610528,0.649316,0.832453,Qwen/Qwen3-0.6B,gsm8k,35
760,761,Joan is at the grocery store. She has a total ...,5,5,"<think>\nOkay, let's see. Joan wants to buy so...",-0.259715,0.000000,885,1,0.76,...,1.00,0.958991,0.938750,0.938750,0.816588,0.832453,0.936061,Qwen/Qwen3-0.6B,gsm8k,35
217,218,Ravi has some coins. He has 2 more quarters th...,350,3.50,"<think>\nOkay, let's see. So Ravi has some coi...",-0.190328,-0.110765,772,0,0.63,...,0.00,0.764609,0.765652,0.938750,1.000000,0.936061,0.936061,Qwen/Qwen3-0.6B,gsm8k,35
988,990,Grace started her own landscaping business. Sh...,567,567,"<think>\nOkay, let's see. Grace has a landscap...",-0.195661,-0.000037,1351,1,0.93,...,1.00,0.861200,0.859474,0.910000,0.856176,0.870357,0.870357,Qwen/Qwen3-0.6B,gsm8k,35
911,913,A trader made a profit of $960 after a week of...,180,180,"<think>\nOkay, let's try to figure out how muc...",-0.210819,-0.000003,581,1,0.93,...,1.00,1.000000,0.942667,0.942667,0.507751,0.557778,0.832453,Qwen/Qwen3-0.6B,gsm8k,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567,570,what is william taft famous for?,['President of the United States'],William Taft is famous for being the only pers...,<think>\nWe are being asked what William Taft ...,-0.178161,-0.075105,403,0,0.08,...,0.99,0.000000,0.085667,0.085667,0.000000,0.010000,0.020218,nvidia/Nemotron-Cascade-8B-Thinking,webq,38
891,895,when did the detroit pistons last win the cham...,['2004 NBA Finals'],2004,"<think>\nWe are being asked: ""When did the Det...",-0.094861,0.000000,339,1,0.65,...,1.00,0.740069,0.562000,0.562000,0.451039,0.402666,0.396977,nvidia/Nemotron-Cascade-8B-Thinking,webq,38
865,869,where is ancient phoenician?,['Lebanon'],"\text{Lebanon, Syria, and parts of Israel",<think>\nWe are being asked about the location...,-0.112417,-0.068436,408,1,0.01,...,0.85,0.287075,0.277273,0.277196,0.000000,0.036237,0.046667,nvidia/Nemotron-Cascade-8B-Thinking,webq,38
876,880,where did charles darwin die?,['Down House'],"\text{Down House, Kent, England",<think>\nWe are being asked where Charles Darw...,-0.107405,-0.117779,235,1,0.40,...,0.99,0.534468,0.450000,0.562000,0.472951,0.410000,0.295000,nvidia/Nemotron-Cascade-8B-Thinking,webq,38


In [16]:
full_df.to_csv("../outputs/experiment_results.csv", index=False)

In [66]:
full_df.columns

Index(['id', 'question', 'ground_truth', 'answer', 'response', 'avg_logprobs',
       'ans_logprobs', 'num_output_tokens', 'correct', 'consistency',
       'embeddings', 'alt_embeddings', 'question_embeddings', 'token_probs',
       'ans_token_probs', 'ridge_clip', 'split_isotonic_on_ridge',
       'split_isotonic_on_ridge_nrt', 'vc', 'blackbox_ridge_clip',
       'blackbox_split_isotonic_on_ridge',
       'blackbox_split_isotonic_on_ridge_nrt', 'question_only_ridge_clip',
       'question_only_split_isotonic_on_ridge',
       'question_only_split_isotonic_on_ridge_nrt', 'model', 'dataset',
       'seed'],
      dtype='str')

In [67]:
full_df.iloc[0]

id                                                                                         271
question                                     Nadia was sent to the flower shop to buy 20 ro...
ground_truth                                                                               250
answer                                                                                     250
response                                     <think>\nOkay, let's see. Nadia wants to buy 2...
avg_logprobs                                                                         -0.215104
ans_logprobs                                                                         -0.000002
num_output_tokens                                                                          655
correct                                                                                      1
consistency                                                                                1.0
embeddings                                   [-0.0

In [68]:
full_df.head()

,id,question,ground_truth,answer,response,avg_logprobs,ans_logprobs,num_output_tokens,correct,consistency,...,vc,blackbox_ridge_clip,blackbox_split_isotonic_on_ridge,blackbox_split_isotonic_on_ridge_nrt,question_only_ridge_clip,question_only_split_isotonic_on_ridge,question_only_split_isotonic_on_ridge_nrt,model,dataset,seed
270,271,Nadia was sent to the flower shop to buy 20 ro...,250,250,"<think>\nOkay, let's see. Nadia wants to buy 2...",-0.215104,-2.026555e-06,655,1,1.00,...,1.0,0.835768,0.836563,0.933871,0.693282,0.820833,0.881154,Qwen/Qwen3-0.6B,gsm8k,0
725,726,"Jed is 10 years older than Matt. In 10 years, ...",20,20,"<think>\nOkay, let's see. So the problem says ...",-0.218885,-2.026556e-06,868,1,1.00,...,1.0,1.000000,0.954706,0.933871,0.929013,0.912736,0.912736,Qwen/Qwen3-0.6B,gsm8k,0
31,31,Noah is a painter. He paints pictures and sell...,1200,1200,"<think>\nOkay, let's see. Noah is a painter wh...",-0.302492,-6.596224e-06,892,1,0.80,...,1.0,0.891935,0.891667,0.836563,1.000000,0.912736,0.912736,Qwen/Qwen3-0.6B,gsm8k,0
342,343,Kim's TV uses 125 watts of electricity per hou...,49,49,"<think>\nOkay, let's see. So the problem is ab...",-0.245416,-9.536739e-07,794,1,0.94,...,1.0,0.944007,0.933871,0.836563,1.000000,0.912736,0.842308,Qwen/Qwen3-0.6B,gsm8k,0
604,605,Ivan has 20 dice. Jerry has twice as many dice...,60,60,"<think>\nOkay, let's see. So the problem says ...",-0.198061,-1.311301e-06,301,1,1.00,...,1.0,1.000000,0.993450,0.994816,1.000000,0.912736,0.912736,Qwen/Qwen3-0.6B,gsm8k,0


In [69]:
full_df["model"].unique()

<ArrowStringArray>
[                         'Qwen/Qwen3-0.6B',
                          'Qwen/Qwen3-1.7B',
              'Qwen/Qwen3-4B-Thinking-2507',
                            'Qwen/Qwen3-8B',
                           'Qwen/Qwen3-14B',
         'nvidia/OpenReasoning-Nemotron-7B',
      'nvidia/Nemotron-Cascade-8B-Thinking',
 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
  'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B']
Length: 9, dtype: str

In [70]:
full_df["dataset"].unique()

<ArrowStringArray>
['gsm8k', 'polymath', 'trivia_qa', 'sciq', 'webq']
Length: 5, dtype: str